In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [2]:
import os

DATA_PATH = "/kaggle/input/competitions/hack-rush-2026-phase-2-few-shot-bone-marrow-challenge"

TRAIN_PATH = os.path.join(DATA_PATH, "phase2_support/phase2_support")
TEST_PATH = os.path.join(DATA_PATH, "phase2_test_20/phase2_test_20")

print("Train folders:", os.listdir(TRAIN_PATH)[:5])
print("Test sample:", os.listdir(TEST_PATH)[:5])

Train folders: ['disease18', 'disease15', 'disease20', 'disease19', 'disease17']
Test sample: ['cp2_00032.jpg', 'cp2_00078.jpg', 'cp2_00137.jpg', 'cp2_00057.jpg', 'cp2_00052.jpg']


In [3]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [4]:
train_dataset = datasets.ImageFolder(
    TRAIN_PATH,
    transform=transform 
)

print("Total images:", len(train_dataset))
print("Classes:", train_dataset.classes)

Total images: 30
Classes: ['disease15', 'disease16', 'disease17', 'disease18', 'disease19', 'disease20']


In [5]:
batch_size = 8
num_workers = 0

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
)

In [ ]:
model = models.efficientnet_b4()

model.classifier[1] = nn.Linear(model.classifier[1].in_features, 14)

weights = torch.load(
    "/kaggle/input/datasets/chandakshubham/phase1-b4-weights/sub.pth",
    map_location=device
)

model.load_state_dict(weights)

print("Phase-1 weights loaded")

model.classifier[1] = nn.Linear(model.classifier[1].in_features, 6)

model = model.to(device)

print("Phase-2 classifier ready")


Phase-1 weights loaded
Phase-2 classifier ready


In [7]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=30,
    eta_min=1e-6
)

scaler = torch.amp.GradScaler("cuda")

In [ ]:
EPOCHS = 30

train_losses = []
train_accuracies = []

for epoch in range(EPOCHS):

    if epoch == 3:
        print("Unfreezing backbone...")
        for param in model.parameters():
            param.requires_grad = True

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda"):

            outputs = model(images)

            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

        preds = torch.argmax(outputs, dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(train_loader)

    accuracy = correct / total

    train_losses.append(avg_loss)
    train_accuracies.append(accuracy)

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | Accuracy: {accuracy:.4f}")

    scheduler.step()

Epoch 1/30 | Loss: 1.9407 | Accuracy: 0.1000
Epoch 2/30 | Loss: 1.8909 | Accuracy: 0.1333
Epoch 3/30 | Loss: 1.4357 | Accuracy: 0.5333
Unfreezing backbone...
Epoch 4/30 | Loss: 1.1984 | Accuracy: 0.6333
Epoch 5/30 | Loss: 1.0564 | Accuracy: 0.7667
Epoch 6/30 | Loss: 0.6803 | Accuracy: 0.8333
Epoch 7/30 | Loss: 0.5255 | Accuracy: 1.0000
Epoch 8/30 | Loss: 0.3295 | Accuracy: 0.9667
Epoch 9/30 | Loss: 0.3227 | Accuracy: 1.0000
Epoch 10/30 | Loss: 0.1564 | Accuracy: 1.0000
Epoch 11/30 | Loss: 0.2023 | Accuracy: 1.0000
Epoch 12/30 | Loss: 0.0661 | Accuracy: 1.0000
Epoch 13/30 | Loss: 0.0830 | Accuracy: 1.0000
Epoch 14/30 | Loss: 0.0431 | Accuracy: 1.0000
Epoch 15/30 | Loss: 0.0253 | Accuracy: 1.0000
Epoch 16/30 | Loss: 0.0621 | Accuracy: 1.0000
Epoch 17/30 | Loss: 0.0306 | Accuracy: 1.0000
Epoch 18/30 | Loss: 0.0153 | Accuracy: 1.0000
Epoch 19/30 | Loss: 0.0098 | Accuracy: 1.0000
Epoch 20/30 | Loss: 0.0222 | Accuracy: 1.0000
Epoch 21/30 | Loss: 0.0194 | Accuracy: 1.0000
Epoch 22/30 | Loss: 

In [9]:
torch.save(model.state_dict(), "/kaggle/working/sub_phase2.pth")

print("Phase-2 model saved")

Phase-2 model saved


In [10]:
model.eval()

class_names = train_dataset.classes
test_images = sorted(os.listdir(TEST_PATH))

image_names = []
predictions = []

with torch.no_grad():

    for img_name in test_images:

        img_path = os.path.join(TEST_PATH, img_name)

        image = Image.open(img_path).convert("RGB")

        image = transform(image).unsqueeze(0).to(device)

        output = model(image)

        pred = torch.argmax(output, dim=1).item()

        predicted_class = class_names[pred]

        image_names.append(img_name)
        predictions.append(predicted_class)

In [11]:
submission = pd.DataFrame({
    "image_name": image_names,
    "predicted_class": predictions
})

submission.to_csv("/kaggle/working/sub_phase2.csv", index=False)

print("Submission file saved!")
submission.head()

Submission file saved!


,image_name,predicted_class
0,cp2_00001.jpg,disease15
1,cp2_00002.jpg,disease15
2,cp2_00003.jpg,disease15
3,cp2_00004.jpg,disease15
4,cp2_00005.jpg,disease18
